In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import os
import time
from datetime import datetime

# ======================================================
# CONFIGURATION
# ======================================================
SAVE_DIR = r"C:\Users\JamJayDatuin\Documents\Machine Learning Projects\SignLanguagesDataset\dataset\ASL"
os.makedirs(SAVE_DIR, exist_ok=True)

MAX_FRAMES_LIMIT = 350   # Auto-stop limit
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

holistic = mp_holistic.Holistic(
    static_image_mode=False,
    model_complexity=1,
    smooth_landmarks=True,
    refine_face_landmarks=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# ======================================================
# Extract BOTH HANDS only (126 features)
# ======================================================
def extract_keypoints(results):
    lh = np.zeros(21 * 3)
    rh = np.zeros(21 * 3)

    if results.left_hand_landmarks:
        lh = np.array([[lm.x, lm.y, lm.z]
                       for lm in results.left_hand_landmarks.landmark]).flatten()

    if results.right_hand_landmarks:
        rh = np.array([[lm.x, lm.y, lm.z]
                       for lm in results.right_hand_landmarks.landmark]).flatten()

    return np.concatenate([lh, rh])


# ======================================================
# MAIN COLLECTION LOOP
# ======================================================
def record_sample(label):
    label_dir = os.path.join(SAVE_DIR, label)
    os.makedirs(label_dir, exist_ok=True)

    # SINGLE CSV PER LABEL
    final_csv = os.path.join(label_dir, f"{label}.csv")

    print(f"\n🎬 Recording sign: {label}")
    print("Starting in 3 seconds...")
    time.sleep(3)

    cap = cv2.VideoCapture(0)
    frames = []
    frame_count = 0

    print("👉 Recording NOW! Press ESC to stop.\n")

    while True:
        ret, frame = cap.read()
        if not ret:
            continue

        frame = cv2.flip(frame, 1)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = holistic.process(rgb)

        hands_detected = results.left_hand_landmarks or results.right_hand_landmarks

        if hands_detected:
            keypoints = extract_keypoints(results)
            frames.append(keypoints)
            frame_count += 1

            # Draw hands
            if results.left_hand_landmarks:
                mp_drawing.draw_landmarks(frame, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
            if results.right_hand_landmarks:
                mp_drawing.draw_landmarks(frame, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)

        else:
            cv2.putText(frame, "NO HANDS DETECTED", (10, 160),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)

        # UI overlays
        cv2.putText(frame, f"Label: {label}", (10, 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.putText(frame, f"Frames Saved: {frame_count}", (10, 75),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
        cv2.putText(frame, "Press ESC to stop & save", (10, 115),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 200, 255), 2)
        cv2.putText(frame, f"Auto-stop at {MAX_FRAMES_LIMIT} frames", (10, 155),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 0), 2)

        cv2.imshow("Dynamic Sign Recorder", frame)

        key = cv2.waitKey(1)

        # Manual stop
        if key == 27:
            print("🛑 ESC pressed, stopping recording...")
            break

        # Automatic stop
        if frame_count >= MAX_FRAMES_LIMIT:
            print(f"⏳ Auto-stop: {MAX_FRAMES_LIMIT} frames reached.")
            break

    cap.release()
    cv2.destroyAllWindows()

    # ======================================================
    # APPEND OR CREATE DATASET
    # ======================================================
    new_df = pd.DataFrame(frames)
    new_df.insert(0, "frame_index", range(len(frames)))

    if os.path.isfile(final_csv):
        print("📄 Existing dataset found — appending new samples...")
        old_df = pd.read_csv(final_csv)

        # Reindex frame_index when appending
        new_df["frame_index"] = np.arange(len(old_df), len(old_df) + len(new_df))

        combined_df = pd.concat([old_df, new_df], ignore_index=True)
        combined_df.to_csv(final_csv, index=False)

        print(f"✔ Updated dataset → {final_csv}")
        print(f"📦 Total frames in dataset: {combined_df.shape[0]}")

    else:
        print("📄 No existing dataset. Creating new dataset file...")
        new_df.to_csv(final_csv, index=False)
        print(f"✔ Created dataset → {final_csv}")

    print(f"📦 Frames added this session: {len(new_df)}")


# ======================================================
# USER INTERFACE
# ======================================================
if __name__ == "__main__":
    print("\n=== Dynamic Sign Language Dataset Collector ===")
    print("Collect dynamic sign sequences using BOTH HANDS only.")
    print(f"Auto-stop triggers at {MAX_FRAMES_LIMIT} frames.\n")

    while True:
        label = input("Label: ").strip().upper()

        if not label:
            print("Invalid label.")
            continue

        record_sample(label)



=== Dynamic Sign Language Dataset Collector ===
Collect dynamic sign sequences using BOTH HANDS only.
Pauses automatically when no hands are detected.
Auto-stops at 350 frames.


🎬 Recording sign: SLEEP
Starting in 3 seconds...
👉 Recording NOW! Press ESC to stop recording.

🛑 ESC pressed, stopping recording...
✔ Saved sample → C:\Users\JamJayDatuin\Documents\Machine Learning Projects\SignLanguagesDataset\dataset\ASL\SLEEP\SLEEP_20251205_174515.csv
📦 Total frames collected: 47

🎬 Recording sign: SLEEP
Starting in 3 seconds...
👉 Recording NOW! Press ESC to stop recording.

⏳ Auto-stop: 350 frames reached. Saving...
✔ Saved sample → C:\Users\JamJayDatuin\Documents\Machine Learning Projects\SignLanguagesDataset\dataset\ASL\SLEEP\SLEEP_20251205_174635.csv
📦 Total frames collected: 350

🎬 Recording sign: HOUSE
Starting in 3 seconds...
👉 Recording NOW! Press ESC to stop recording.

⏳ Auto-stop: 350 frames reached. Saving...
✔ Saved sample → C:\Users\JamJayDatuin\Documents\Machine Learning Pr

In [12]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.utils import to_categorical

DATASET_DIR = r"C:\Users\JamJayDatuin\Documents\Machine Learning Projects\SignLanguagesDataset\dataset\ASL"
SAVE_DIR = r"C:\Users\JamJayDatuin\Documents\Machine Learning Projects\SignLanguagesDataset\processed\ASL"
os.makedirs(SAVE_DIR, exist_ok=True)

MAX_FRAMES = 40               # Fixed sequence length
FEATURES = 126                # 21 keypoints × 2 hands × 3 coords

X, y = [], []
labels = sorted(os.listdir(DATASET_DIR))
label_map = {label: i for i, label in enumerate(labels)}

print("Found labels:", labels)

def fix_length(seq):
    """Pad or trim recording to MAX_FRAMES."""
    if len(seq) > MAX_FRAMES:
        return seq[:MAX_FRAMES]
    pad = np.zeros((MAX_FRAMES - seq.shape[0], FEATURES))
    return np.vstack([seq, pad])

for label in labels:
    label_folder = os.path.join(DATASET_DIR, label)

    for file in os.listdir(label_folder):
        if not file.endswith(".csv"):
            continue

        df = pd.read_csv(os.path.join(label_folder, file))
        seq = df.drop(columns=["frame_index"]).values  # shape: (frames, 126)

        seq = fix_length(seq)
        X.append(seq)
        y.append(label_map[label])

X = np.array(X)                           # (samples, 40, 126)
y = to_categorical(y, num_classes=len(labels))

print("X shape:", X.shape)
print("y shape:", y.shape)

np.save(os.path.join(SAVE_DIR, "X.npy"), X)
np.save(os.path.join(SAVE_DIR, "y.npy"), y)
np.save(os.path.join(SAVE_DIR, "labels.npy"), labels)

print("\n✅ Dataset processed successfully!")


Found labels: ['BOOK', 'CAR', 'FAMILY', 'HAPPY', 'HELLO', 'HOUSE', 'HOW', 'I_LOVE_YOU', 'NO', 'PHONE', 'PLAY', 'READ', 'SLEEP', 'THANKYOU', 'TIRED', 'WATER', 'WHY', 'WORK', 'WRITE', 'YES']
X shape: (20, 40, 126)
y shape: (20, 20)

✅ Dataset processed successfully!


In [13]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import os

DATA_DIR = r"C:\Users\JamJayDatuin\Documents\Machine Learning Projects\SignLanguagesDataset\processed\ASL"
MODEL_DIR = r"C:\Users\JamJayDatuin\Documents\Machine Learning Projects\SignLanguagesDataset\models\ASL"
os.makedirs(MODEL_DIR, exist_ok=True)

X = np.load(f"{DATA_DIR}/X.npy")
y = np.load(f"{DATA_DIR}/y.npy")
labels = np.load(f"{DATA_DIR}/labels.npy", allow_pickle=True)

SEQUENCE_LEN = X.shape[1]     # 40
FEATURES = X.shape[2]         # 126
NUM_CLASSES = y.shape[1]

print("Training data:", X.shape, y.shape)

model = models.Sequential([
    layers.Masking(mask_value=0., input_shape=(SEQUENCE_LEN, FEATURES)),
    layers.Bidirectional(layers.LSTM(128, return_sequences=True)),
    layers.Bidirectional(layers.LSTM(64)),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X, y,
    epochs=50,
    batch_size=16,
    validation_split=0.2
)

model_path = os.path.join(MODEL_DIR, "asl_dynamic_lstm.keras")
model.save(model_path)

print("\n🎉 Training complete!")
print("Saved model →", model_path)


Training data: (20, 40, 126) (20, 20)
Epoch 1/50
1/1 [==============================] - 24s 24s/step - loss: 3.0541 - accuracy: 0.0625 - val_loss: 3.1583 - val_accuracy: 0.0000e+00
Epoch 2/50
1/1 [==============================] - 0s 411ms/step - loss: 2.9091 - accuracy: 0.1250 - val_loss: 3.2956 - val_accuracy: 0.0000e+00
Epoch 3/50
1/1 [==============================] - 0s 414ms/step - loss: 2.8830 - accuracy: 0.0625 - val_loss: 3.4071 - val_accuracy: 0.0000e+00
Epoch 4/50
1/1 [==============================] - 0s 412ms/step - loss: 2.8320 - accuracy: 0.1250 - val_loss: 3.5090 - val_accuracy: 0.0000e+00
Epoch 5/50
1/1 [==============================] - 0s 414ms/step - loss: 2.7886 - accuracy: 0.1875 - val_loss: 3.5859 - val_accuracy: 0.0000e+00
Epoch 6/50
1/1 [==============================] - 0s 419ms/step - loss: 2.7582 - accuracy: 0.1875 - val_loss: 3.6341 - val_accuracy: 0.0000e+00
Epoch 7/50
1/1 [==============================] - 0s 418ms/step - loss: 2.8051 - accuracy: 0.1250 -

In [14]:
import tensorflow as tf
import numpy as np
import os

# Correct paths
MODEL_DIR = r"C:\Users\JamJayDatuin\Documents\Machine Learning Projects\SignLanguagesDataset\models\ASL"
os.makedirs(MODEL_DIR, exist_ok=True)

model_path = os.path.join(MODEL_DIR, "asl_dynamic_lstm.keras")
print("Loading model from:", model_path)

if not os.path.isfile(model_path):
    raise FileNotFoundError(f"❌ Model not found at: {model_path}")

model = tf.keras.models.load_model(model_path)

# ================================
# ⭐ FIX THAT SOLVES LSTM CONVERSION PROBLEM
# ================================
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# required when using LSTM / Bidirectional
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]

# disable TensorList lowering (critical fix)
converter._experimental_lower_tensor_list_ops = False

# enable optimizations (optional)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# ================================
# Convert
# ================================
tflite_model = converter.convert()

# Save final TFLite model
tflite_path = os.path.join(MODEL_DIR, "asl_dynamic_lstm.tflite")

with open(tflite_path, "wb") as f:
    f.write(tflite_model)

print("\n✔ Exported TFLite model successfully!")
print("Saved →", tflite_path)


Loading model from: C:\Users\JamJayDatuin\Documents\Machine Learning Projects\SignLanguagesDataset\models\ASL\asl_dynamic_lstm.keras


INFO:tensorflow:Assets written to: C:\Users\JAMJAY~1\AppData\Local\Temp\tmpqqnyicxn\assets


INFO:tensorflow:Assets written to: C:\Users\JAMJAY~1\AppData\Local\Temp\tmpqqnyicxn\assets



✔ Exported TFLite model successfully!
Saved → C:\Users\JamJayDatuin\Documents\Machine Learning Projects\SignLanguagesDataset\models\ASL\asl_dynamic_lstm.tflite


In [ ]:
import cv2
import numpy as np
import tensorflow as tf
from collections import deque
import mediapipe as mp
import os

# ==========================
# PATHS
# ==========================
BASE = r"C:\Users\JamJayDatuin\Documents\Machine Learning Projects\SignLanguagesDataset"
MODEL_DIR = os.path.join(BASE, "models", "ASL")
PROCESSED_DIR = os.path.join(BASE, "processed", "ASL")

MODEL_PATH = os.path.join(MODEL_DIR, "asl_dynamic_lstm.keras")
LABELS_PATH = os.path.join(PROCESSED_DIR, "labels.npy")

MAX_FRAMES = 40
FEATURES = 126
SMOOTHING_WINDOW = 8       # number of predictions to average
CONFIDENCE_THRESHOLD = 0.50

# ==========================
# LOAD MODEL & LABELS
# ==========================
labels = np.load(LABELS_PATH, allow_pickle=True)
model = tf.keras.models.load_model(MODEL_PATH)

print("Loaded model:", MODEL_PATH)
print("Loaded labels:", labels)

# ==========================
# FAST MEDIAPIPE HAND LANDMARKER
# ==========================
BaseOptions = mp.tasks.BaseOptions
VisionRunningMode = mp.tasks.vision.RunningMode
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions

options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path="hand_landmarker.task"),
    running_mode=VisionRunningMode.IMAGE,
    num_hands=2
)
landmarker = HandLandmarker.create_from_options(options)

# ==========================
# HELPER FUNCTIONS
# ==========================
def extract_hand_points(result):
    """Extract 126 hand keypoints (or zeros if missing)."""
    lh = np.zeros(21 * 3)
    rh = np.zeros(21 * 3)

    if result.hand_landmarks:
        if len(result.hand_landmarks) > 0:
            lh = np.array([[lm.x, lm.y, lm.z] for lm in result.hand_landmarks[0]]).flatten()

        if len(result.hand_landmarks) > 1:
            rh = np.array([[lm.x, lm.y, lm.z] for lm in result.hand_landmarks[1]]).flatten()

    return np.concatenate([lh, rh])


def smooth_predictions(history):
    """Average softmax outputs to stabilize predictions."""
    if len(history) == 0:
        return None, 0.0

    avg_probs = np.mean(history, axis=0)
    pred_idx = int(np.argmax(avg_probs))
    confidence = float(avg_probs[pred_idx])
    return pred_idx, confidence


# ==========================
# LIVE CAMERA LOOP
# ==========================
sequence = deque(maxlen=MAX_FRAMES)
smooth_history = deque(maxlen=SMOOTHING_WINDOW)

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        continue

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = landmarker.detect(mp_image)

    # HAND DETECTED
    if result.hand_landmarks:
        keypoints = extract_hand_points(result)


Loaded labels: ['BOOK' 'CAR' 'FAMILY' 'HAPPY' 'HELLO' 'HOUSE' 'HOW' 'I_LOVE_YOU' 'NO'
 'PHONE' 'PLAY' 'READ' 'SLEEP' 'THANKYOU' 'TIRED' 'WATER' 'WHY' 'WORK'
 'WRITE' 'YES']
Loaded model: C:\Users\JamJayDatuin\Documents\Machine Learning Projects\SignLanguagesDataset\models\ASL\asl_dynamic_lstm.keras
